## Setup Colab and sync


In [1]:
#@title Clone the repo and add API key
!git clone https://github.com/amusali/bert_for_patents.git
import os
# Api key setting
os.environ['patentsview_api_key'] = 'Uch9R1RR.BxaQcxV8HK4nBMgTTjX1F9CQW8PF4nXw'  # Replace with your actual key

Cloning into 'bert_for_patents'...
remote: Enumerating objects: 879, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 879 (delta 14), reused 13 (delta 13), pack-reused 854 (from 4)
Receiving objects: 100% (879/879), 159.23 MiB | 54.42 MiB/s, done.
Resolving deltas: 100% (479/479), done.


In [2]:
#@title Pull latest changes from repo
%cd "/content/bert_for_patents"
!git pull origin master
#@title Add paths to the system and change directory
import sys
sys.path.append('/content/bert_for_patents/05 Analysis/01 Main')
os.chdir("/content/bert_for_patents/05 Analysis/01 Main")

/content/bert_for_patents
From https://github.com/amusali/bert_for_patents
 * branch            master     -> FETCH_HEAD
Already up to date.


In [3]:
#@title Run setup file to get environment
!python "/content/bert_for_patents/05 Analysis/01 Main/setup_colab.py"

import sys
sys.path.append('/content/bert_for_patents/05 Analysis/01 Main')
os.chdir("/content/bert_for_patents/05 Analysis/01 Main")

/content/bert_for_patents/05 Analysis/01 Main/setup_colab.py:10: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources
absl-py==2.1.0 is already installed.
Installing asttokens==2.4.1...
astunparse==1.6.3 is already installed.
attrs==23.2.0 is already installed.
beautifulsoup4==4.12.3 is already installed.
Installing bert==2.2.0...
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Created wheel for bert: filename=bert-2.2.0-py3-none-any.whl size=3745 sha256=ef0c03f8b85b861d2f63070989c7223e0cf4766d620eeaaae4675bd530d20161
  Stored in directory: /root/.cache/pip/wheels/55/82/8d/a9bad0b8280eb858aa3dcb4e617ee5a1653fdeb239e1e8c3fe
  Created wheel for erlastic: filename=erlastic-2.0.0-py3-none-any.whl size=6780 sha256=a12aea1c8a23431f4883f80da449f223e68d06dfe197c08da24bb1859bd29f39
  Stored in directory: /root/.cache/pip/wheels/63/ea/24/ab8ff86604f1a87ca69a06af89bb7e080a5

In [6]:
#@title Mount Drive to get BERT model
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


Set precision

## Load the patents and process

In [ ]:
#@title Define classes and functions
import time
import pandas as pd
from torch.utils.data import Dataset, DataLoader
#import api.bertembeddings as be
#import importlib
#importlib.reload(be)
import os
!pip install dill
import dill as pickle
from datetime import datetime
import numpy as np

# Custom Dataset class that returns the raw abstract text
class PatentDataset(Dataset):
    def __init__(self, df):
        """
        df: A Pandas DataFrame containing at least a column 'patent_abstract'.
        """
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        # Return a tuple of (patent_id, abstract)
        patent_id = self.df.iloc[idx]['patent_id'] # Assumes a 'patent_id' column exists
        abstract = self.df.iloc[idx]['abstract']
        return patent_id, abstract

# Custom collate function to process a batch of abstracts
def collate_fn(batch):
    """
    Receives a list of tuples (patent_id, abstract).
    Processes the abstracts in batch using get_embd_of_whole_abstract.
    Returns:
      - A list of patent_ids.
      - An array of embeddings with shape (batch_size, 1024)
    """
    patent_ids, abstracts = zip(*batch)
    embeddings = be.get_embd_of_whole_abstract(list(abstracts), return_cls_embedding=True, has_context_token=True)
    return list(patent_ids), embeddings


def save_embedding_dict(embedding_dict, folder):
    """
    Saves the embeddings dictionary to a pickle file with a timestamped filename.
    """
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    file_name = f"01 CLS embeddings rest - {timestamp}.pkl"
    file_path = os.path.join(folder, file_name)
    with open(file_path, "wb") as f:
        pickle.dump(embedding_dict, f)
    print(f"Saved embeddings to {file_path}")

def load(filename):
    with open(filename, 'rb') as f:
        return pickle.load(f)




In [ ]:
#@title Load data and process & save every 90 minutes
import dill as pickle

# Load data from Drive
path = '/content/drive/MyDrive/PhD Data/06 Patents to be fed to BERT/patents_to_check.pkl'
df = load(path)

dataset = PatentDataset(df)
dataloader = DataLoader(dataset, batch_size=32, collate_fn=collate_fn, shuffle=False)

SAVE_FOLDER = '/content/drive/MyDrive/PhD Data/01 CLS Embeddings'
embedding_dict = {}       # Dictionary to store patent_id -> (1024,) embedding
save_interval = 90 * 60     # 90 minutes in seconds
last_save_time = time.time()

# Variables for progress tracking
start_time = time.time()
last_progress_time = start_time
processed_count = 0
processed_in_last_minute = 0


In [ ]:
#@title Main loop
from datetime import datetime



for batch_patent_ids, batch_embeddings in dataloader:
    # Process each patent in the batch.
    for patent_id, embedding in zip(batch_patent_ids, batch_embeddings):
        # Ensure each embedding is of shape (1024,)
        embedding = np.squeeze(embedding, axis=0) if embedding.ndim == 2 else embedding
        embedding_dict[patent_id] = embedding
        processed_count += 1
        processed_in_last_minute += 1

    # Current time after processing the batch
    current_time = time.time()

    # If one minute has passed, print progress info
    if current_time - last_progress_time >= 60:
        print(f"In the last minute, processed {processed_in_last_minute} patents. Total processed: {processed_count}.")
        average_time = (current_time - last_progress_time) / processed_in_last_minute
        print(f"Average processing per patent: {average_time:.5f} ")
        last_progress_time = current_time
        processed_in_last_minute = 0

    # Save the dictionary every 90 minutes.
    if current_time - last_save_time >= save_interval:
        save_embedding_dict(embedding_dict, folder=SAVE_FOLDER)
        last_save_time = current_time

# Final save after processing is complete.
save_embedding_dict(embedding_dict, folder=SAVE_FOLDER)

In [ ]:
!pip install dill

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.4/119.4 kB 10.8 MB/s eta 0:00:00


In [ ]:
print(len(embedding_dict))

6997638


# Combine with previously processed patents

In [ ]:
#@title Load all patents
previous_patents = load(r"/content/drive/MyDrive/PhD Data/01 CLS Embeddings/CheckedPatents_CLSonly_2024.11.11_00:06.pkl")
current_patents = load(r"/content/drive/MyDrive/PhD Data/01 CLS Embeddings/01 CLS embeddings rest - 20250226_163344.pkl")

assert(len(previous_patents) == 2077783)
assert(len(current_patents) == 6997638)


In [ ]:
#@title Combine, check and save

# Squeeze
for key, value in previous_patents.items():
    previous_patents[key] = value.squeeze()


# Combine
current_patents.update(previous_patents)
previous_patents = None



In [ ]:
# Get a list of all variables
all_vars = list(globals().keys())
print(all_vars)


['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh', 'In', 'Out', 'get_ipython', 'exit', 'quit', '_', '__', '___', '_i', '_ii', '_iii', '_i1', 'drive', '_i2', '_i3', 'time', 'pd', 'Dataset', 'DataLoader', 'os', '_exit_code', 'pickle', 'datetime', 'np', 'PatentDataset', 'collate_fn', 'save_embedding_dict', 'load', '_i4', 'previous_patents', 'current_patents', '_i5', 'key', 'value', '_i6', '_6', '_i7', '_7', '_i8', 'gc', 'sys', 'get_memory_variables', 'name', 'var', '_i9']


In [ ]:


#@title Check

#assert(len(current_patents) == 2777783 + 6997638)
print(len(current_patents))

# Check types
size_types  = set()
for key, value in current_patents.items():
    size_types.add(value.shape)

print(size_types)
assert(len(size_types) == 1)

9075421
{(1024,)}


In [ ]:
import dill as pickle
import os

# Define batch size
num_batches = 16  # Adjust based on testing
batch_size = len(current_patents) // num_batches

# Define save path (Change to your Google Drive path if using Colab)
save_dir = "/content/drive/MyDrive/PhD Data/01 CLS Embeddings/Batched embeddings"
os.makedirs(save_dir, exist_ok=True)

# Save in batches
for i in range(num_batches):
    start = i * batch_size
    end = (i + 1) * batch_size if i != num_batches - 1 else len(current_patents)

    batch_dict = dict(list(current_patents.items())[start:end])

    with open(f"{save_dir}/batch_{i}.pkl", "wb") as f:
        pickle.dump(batch_dict, f, protocol=pickle.HIGHEST_PROTOCOL)

    print(f"Saved batch {i+1}/{num_batches} - Size: {len(batch_dict)} items")


Saved batch 1/16 - Size: 567213 items
Saved batch 2/16 - Size: 567213 items
Saved batch 3/16 - Size: 567213 items
Saved batch 4/16 - Size: 567213 items
Saved batch 5/16 - Size: 567213 items
Saved batch 6/16 - Size: 567213 items
Saved batch 7/16 - Size: 567213 items
Saved batch 8/16 - Size: 567213 items
Saved batch 9/16 - Size: 567213 items
Saved batch 10/16 - Size: 567213 items
Saved batch 11/16 - Size: 567213 items
Saved batch 12/16 - Size: 567213 items
Saved batch 13/16 - Size: 567213 items
Saved batch 14/16 - Size: 567213 items
Saved batch 15/16 - Size: 567213 items
Saved batch 16/16 - Size: 567226 items


# Combine, adjust precision and save to save memory

In [7]:
import os
import glob
import pickle
import numpy as np

# Define the folder containing your batched embeddings.
embedding_batches_folder = '/content/drive/MyDrive/PhD Data/01 CLS Embeddings/Batched embeddings'
# Define the output file path for the combined embeddings.
output_file = '/content/drive/MyDrive/PhD Data/01 CLS Embeddings/All embeddings - float16/all_embeddings_float16.pkl'

combined_embeddings = {}

# Get list of all batch files (assumed to have a .pkl extension)
batch_files = glob.glob(os.path.join(embedding_batches_folder, '*.pkl'))

print(f"Found {len(batch_files)} batch files. Processing...")

for batch_file in batch_files:
    print(f"Loading {batch_file}...")
    with open(batch_file, 'rb') as f:
        batch_dict = pickle.load(f)
    # Ensure keys are strings and values are NumPy arrays.
    for k, v in batch_dict.items():
        combined_embeddings[str(k)] = np.array(v)

print("All batch files loaded. Converting embeddings to float16...")

# Convert every embedding to float16.
for k in combined_embeddings:
    combined_embeddings[k] = combined_embeddings[k].astype(np.float16)

print("Conversion complete. Saving the combined embeddings to file...")

# Save the combined dictionary as a single pickle file.
with open(output_file, 'wb') as f:
    pickle.dump(combined_embeddings, f)

print(f"Combined embeddings saved to {output_file}.")

# Calculate the total size of combined_embeddings in bytes and print it in MB.
total_size_bytes = sum(emb.nbytes for emb in combined_embeddings.values())
print(f"Total size of combined_embeddings: {total_size_bytes / (1024**2):.2f} MB")



Found 16 batch files. Processing...
Loading /content/drive/MyDrive/PhD Data/01 CLS Embeddings/Batched embeddings/batch_0.pkl...
Loading /content/drive/MyDrive/PhD Data/01 CLS Embeddings/Batched embeddings/batch_1.pkl...
Loading /content/drive/MyDrive/PhD Data/01 CLS Embeddings/Batched embeddings/batch_2.pkl...
Loading /content/drive/MyDrive/PhD Data/01 CLS Embeddings/Batched embeddings/batch_3.pkl...
Loading /content/drive/MyDrive/PhD Data/01 CLS Embeddings/Batched embeddings/batch_4.pkl...
Loading /content/drive/MyDrive/PhD Data/01 CLS Embeddings/Batched embeddings/batch_5.pkl...
Loading /content/drive/MyDrive/PhD Data/01 CLS Embeddings/Batched embeddings/batch_6.pkl...
Loading /content/drive/MyDrive/PhD Data/01 CLS Embeddings/Batched embeddings/batch_7.pkl...
Loading /content/drive/MyDrive/PhD Data/01 CLS Embeddings/Batched embeddings/batch_8.pkl...
Loading /content/drive/MyDrive/PhD Data/01 CLS Embeddings/Batched embeddings/batch_9.pkl...
Loading /content/drive/MyDrive/PhD Data/01 C